# Final models: training

Trains and saves the B2 models run on the focal-sampling data:
1. **HydraMultiROCKET (HR) on B2 rotdaily**: the final classifier (HydraMultiROCKET-Daily)
2. **HR on B2 uncorrected and B2 rotbasal**: comparison models
3. **Random Forest (RF) on the 60 summary features of B2 uncorrected**: comparison model

TabPFN B2 uncorrected, also a comparison model, needs the `tabpfn` environment and is trained in `10a_final_models_training_tabpfn`.

All models are trained on every labelled window the cross-validation used (the 2147 B2 windows in `indices_cv_fold_lookup.csv`), with the same settings as the CV in `05d` (HR) and `05a` (RF): class weighting, seed 42, default hyperparameters. The pooled out-of-fold CV predictions are the reported performance, so there is no hold-out set here. Models are saved to `models/final_hr_b2_unc/`, `models/final_hr_b2_rotbasal/`, `models/final_hr_b2_rotdaily/` and `models/final_rf_b2_unc/`.

In [ ]:
import os
import time
from datetime import datetime

import joblib
import numpy as np
import pandas as pd
import tsai.all as ts
from fastai.torch_core import set_seed
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import LabelEncoder

In [ ]:
SEED = 42
N_WINDOWS = 2147  # labelled B2 windows in the fold lookup
FOLD_LOOKUP_PATH = "../data/temp/indices_cv_fold_lookup.csv"
fold_lookup = pd.read_csv(FOLD_LOOKUP_PATH)

## HydraMultiROCKET
The arrays are built with the same `build_ts_arrays` as `05d`, so the final model sees exactly the windows, labels and channel order the CV used. `train_save_hr` trains, saves and reload-checks one HR model, so the uncorrected and corrected models are trained identically.

In [ ]:
def build_ts_arrays(acc, fold_lookup, burst):
    """Build the time-series X array, aligned to the fold lookup (same as 05d).

    Args:
        acc (pd.DataFrame): Raw accelerometer long-format data straight from
            the parquet file (columns: Ind_ID, new_burst, feature, plus
            numeric time columns)
        fold_lookup (pd.DataFrame): Fold lookup table with columns
            burst, Ind_ID, new_burst, burst_id, behaviour, activity, fold
        burst (str): Folder-derived burst name (e.g. "Burst_2"); matched
            against fold_lookup's lowercase burst column

    Returns:
        tuple: (X, merged) where X is an (n_samples, 3, n_features) array and
            merged is a DataFrame with columns id, Ind_ID, new_burst, burst_id,
            behaviour, activity, fold, idx -- merged.loc[i, "idx"] indexes X's
            first axis for sample i.
    """
    m = fold_lookup.loc[
        fold_lookup["burst"] == burst.lower(),
        ["Ind_ID", "new_burst", "burst_id", "behaviour", "activity", "fold"],
    ]

    # Build id directly from Ind_ID/new_burst (the raw parquet has no id column)
    acc = acc.copy()
    acc["new_burst"] = acc["new_burst"].astype(np.int64)
    acc["id"] = acc["Ind_ID"].astype(str) + "_" + acc["new_burst"].astype(str)
    id_mapping = (
        acc[["id", "Ind_ID", "new_burst"]].drop_duplicates().reset_index(drop=True)
    )

    # Inner join drops Remove/unlabelled ids and attaches fold + label values
    merged = id_mapping.merge(m, on=["Ind_ID", "new_burst"], how="inner")
    merged = merged.reset_index(drop=True)
    merged["idx"] = merged.index

    # Identify the feature columns (all numeric columns)
    feature_cols = [col for col in acc.columns if col.isdigit()]
    n_features = len(feature_cols)
    n_samples = len(merged)
    print(f"Number of samples: {n_samples}, number of features: {n_features}")

    X = np.zeros((n_samples, 3, n_features), dtype=np.float64)
    feature_to_idx = {"X": 0, "Y": 1, "Z": 2}

    for i, id_val in enumerate(merged["id"]):
        id_data = acc[acc["id"] == id_val]
        for feature_type, axis_idx in feature_to_idx.items():
            feature_data = id_data[id_data["feature"] == feature_type]
            if len(feature_data) > 0:
                X[i, axis_idx, :] = feature_data[feature_cols].values[0]
            else:
                print(f"Warning: No data found for ID {id_val}, feature {feature_type}")

    return X.astype(np.float64), merged

In [ ]:
def train_save_hr(correction, name):
    """Train HR on every B2 window of one correction type, save it and reload it.

    Args:
        correction (str): Correction type in the parquet file name
            (uncorrected, rotbasal or rotdaily)
        name (str): Model name; files are saved to ../models/<name>/

    Returns:
        Learner: The trained learner
    """
    model_dir = f"../models/{name}"
    acc = pd.read_parquet(
        f"../data/temp/acc_tsai/Burst_2/annotated_acc_tsai_burst_2_{correction}.parquet"
    )
    X, merged = build_ts_arrays(acc, fold_lookup, "Burst_2")
    assert len(merged) == N_WINDOWS, (
        f"Expected {N_WINDOWS} B2 {correction} windows, got {len(merged)}"
    )

    # Every window is used for training. tsai needs a validation set, so it is the
    # training set itself: there is no early stopping, and "valid" only reports
    # training fit.
    all_idx = list(range(len(merged)))
    dsets = ts.TSDatasets(
        X,
        y=merged["behaviour"].to_numpy(dtype="<U20"),
        tfms=[None, [ts.Categorize()]],
        splits=(all_idx, all_idx),
        inplace=True,
    )

    # Same seed, loss weighting and schedule as the CV in 05d
    set_seed(SEED, reproducible=True)
    dls = ts.TSDataLoaders.from_dsets(dsets.train, dsets.valid, bs=[64, 128])
    model = ts.create_model(ts.HydraMultiRocket, dls=dls).cuda()
    learn = ts.Learner(
        dls, model, metrics=[], loss_func=ts.CrossEntropyLossFlat(weight=dls.train.cws)
    )
    t0 = time.time()
    lr_max = learn.lr_find()
    learn.fit_one_cycle(50, lr_max)
    hr_time = time.time() - t0
    print(f"{name}: trained in {hr_time / 60:.1f} min, classes: {list(dls.vocab)}")

    os.makedirs(model_dir, exist_ok=True)
    learn.save_all(
        path=model_dir,
        dls_fname=f"{name}_dls",
        model_fname=f"{name}_model",
        learner_fname=f"{name}_learner",
    )
    # Class index -> name, read by the inference notebook
    pd.DataFrame(
        {"class_index": range(len(dls.vocab)), "class_name": list(dls.vocab)}
    ).to_csv(f"{model_dir}/{name}_class_mapping.csv", index=False)
    pd.DataFrame(
        [
            {
                "model_name": name,
                "model_type": "HydraMultiRocket",
                "training_date": datetime.now().isoformat(),
                "data": f"Burst_2 {correction}",
                "n_windows": len(merged),
                "class_names": list(dls.vocab),
                "class_weighted_loss": True,
                "seed": SEED,
                "epochs": 50,
                "lr_max": lr_max.valley,
                "training_time_seconds": hr_time,
            }
        ]
    ).to_csv(f"{model_dir}/{name}_training_metadata.csv", index=False)
    print(f"Saved HR model to {model_dir}")

    # Reload check: the inference notebook requires every parameter and buffer on CUDA
    hr = ts.load_learner_all(
        path=model_dir,
        dls_fname=f"{name}_dls",
        model_fname=f"{name}_model",
        learner_fname=f"{name}_learner",
    )
    devices = {p.device.type for p in hr.model.parameters()} | {
        b.device.type for b in hr.model.buffers()
    }
    print(f"Reloaded {name}; parameter/buffer devices: {devices}")
    return learn

### B2 uncorrected (comparison)

In [ ]:
learn_hr = train_save_hr("uncorrected", "final_hr_b2_unc")

### B2 rotbasal and rotdaily (rotdaily: final classifier)
Same function and settings as the uncorrected model; only the input file differs.

In [ ]:
for correction in ["rotbasal", "rotdaily"]:
    train_save_hr(correction, f"final_hr_b2_{correction}")

## Random Forest, B2 uncorrected
Same data preparation and settings as `05a` (`RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=42)`).

In [ ]:
RF_DIR, RF_NAME = "../models/final_rf_b2_unc", "final_rf_b2_unc"
ID_COLS = ["Ind_ID", "new_burst", "burst_id", "behaviour", "activity", "fold"]

features = pd.read_parquet(
    "../data/raw/features/Burst_2/annotated_features_burst_2_uncorrected.parquet"
)
m = fold_lookup.loc[fold_lookup["burst"] == "burst_2", ID_COLS]
df = features.merge(m, on=["Ind_ID", "new_burst"], how="inner").reset_index(drop=True)
X_feat = df.drop(columns=ID_COLS)
assert X_feat.shape == (N_WINDOWS, 60), X_feat.shape

label_encoder = LabelEncoder()
y_feat = label_encoder.fit_transform(df["behaviour"])


def make_rf():
    return RandomForestClassifier(
        n_estimators=100, class_weight="balanced", random_state=SEED
    )

In [ ]:
# Check: this environment reproduces 05a's CV result for one fold. 05a ran in
# torch_tabular (scikit-learn 1.4.2); this kernel has a different scikit-learn
# version, which should not change RF but is not guaranteed to.
te = (df["fold"] == 0).to_numpy()
y_pred = make_rf().fit(X_feat[~te], y_feat[~te]).predict(X_feat[te])
ref = pd.read_csv(
    "../data/output/comparison_behaviour/behaviour_features_ml_metrics.csv"
)
ref = ref[
    (ref["burst"] == "Burst_2")
    & (ref["correction_type"] == "uncorrected")
    & (ref["model"] == "RF")
    & (ref["fold"] == 0)
].iloc[0]
print(
    f"Fold 0 accuracy: {accuracy_score(y_feat[te], y_pred):.6f} (05a: {ref['accuracy']:.6f}); "
    f"f1_macro: {f1_score(y_feat[te], y_pred, average='macro', labels=np.arange(len(label_encoder.classes_)), zero_division=0):.6f} "
    f"(05a: {ref['f1_macro']:.6f})"
)

In [ ]:
rf = make_rf()
t0 = time.time()
rf.fit(X_feat, y_feat)
rf_time = time.time() - t0

os.makedirs(RF_DIR, exist_ok=True)
joblib.dump(rf, f"{RF_DIR}/{RF_NAME}_rf_model.joblib")
joblib.dump(label_encoder, f"{RF_DIR}/{RF_NAME}_label_encoder.joblib")
pd.DataFrame(
    [
        {
            "model_name": RF_NAME,
            "model_type": "RandomForest",
            "training_date": datetime.now().isoformat(),
            "data": "Burst_2 uncorrected",
            "n_windows": len(df),
            "n_features": X_feat.shape[1],
            "class_names": list(label_encoder.classes_),
            "class_weight": "balanced",
            "n_estimators": 100,
            "seed": SEED,
            "training_time_seconds": rf_time,
        }
    ]
).to_csv(f"{RF_DIR}/{RF_NAME}_training_metadata.csv", index=False)

# Reload check
rf_reloaded = joblib.load(f"{RF_DIR}/{RF_NAME}_rf_model.joblib")
assert (rf_reloaded.predict(X_feat) == rf.predict(X_feat)).all()
print(f"Saved RF model to {RF_DIR}; classes: {list(label_encoder.classes_)}")